# Data Analysis con Pandas

> Pandas es una librería de Python escrita como extensión de NumPy para manipulación y análisis de datos.

Sitio oficial: [pandas.python.org](https://pandas.pydata.org/)
Documentación Oficial: [pandas.pydata.org/pandas-docs/stable/](https://pandas.pydata.org/pandas-docs

## Tablas Pivot con Pandas

In [1]:
import pandas as pd

Las **tablas pivot** (`pivot_table`) son una herramienta fundamental de Pandas para **resumir datos** en un DataFrame con un propósito específico.

- Las tablas pivot hacen un uso intensivo de las **funciones de agregación**.
- Una tabla pivot es en sí misma un DataFrame, donde las filas representan una variable de interés, las columnas otra, y las celdas contienen algún **valor agregado** (como la media o la suma).
- Una tabla pivot a menudo incluye **valores marginales** (totales generales), los que permiten ver la relación entre dos variables de un vistazo.


### Preparando el Dataset

#### Carga y Exploración del Dataset


Utilizaremos el conjunto de datos de la **Clasificación Mundial de Universidades de Times Higher Education** (Times Higher Education World University Ranking), una de las métricas universitarias más influyentes.

In [2]:
path_to_data = '../data/'
df = pd.read_csv(f'{path_to_data}cwurData.csv')
df

,world_rank,institution,country,national_rank,quality_of_education,alumni_employment,quality_of_faculty,publications,influence,citations,broad_impact,patents,score,year
0,1,Harvard University,USA,1,7,9,1,1,1,1,NaN,5,100.00,2012
1,2,Massachusetts Institute of Technology,USA,2,9,17,3,12,4,4,NaN,1,91.67,2012
2,3,Stanford University,USA,3,17,11,5,4,2,2,NaN,15,89.50,2012
3,4,University of Cambridge,United Kingdom,1,10,24,4,16,16,11,NaN,50,86.17,2012
4,5,California Institute of Technology,USA,4,2,29,7,37,22,22,NaN,18,85.21,2012
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2195,996,University of the Algarve,Portugal,7,367,567,218,926,845,812,969.0,816,44.03,2015
2196,997,Alexandria University,Egypt,4,236,566,218,997,908,645,981.0,871,44.03,2015
2197,998,Federal University of Ceará,Brazil,18,367,549,218,830,823,812,975.0,824,44.03,2015
2198,999,University of A Coruña,Spain,40,367,567,218,886,974,812,975.0,651,44.02,2015


In [3]:
df.dtypes

world_rank                int64
institution              object
country                  object
national_rank             int64
quality_of_education      int64
alumni_employment         int64
quality_of_faculty        int64
publications              int64
influence                 int64
citations                 int64
broad_impact            float64
patents                   int64
score                   float64
year                      int64
dtype: object

#### Creación de la Columna `Rank_Level`


Antes de crear la tabla pivot, vamos a crear una nueva columna llamada **Rank\_Level** para categorizar las universidades según su `world_rank`:

  * **Primer Nivel:** 1-100
  * **Segundo Nivel:** 101-200
  * **Tercer Nivel:** 201-300
  * **Otras Top Universidades:** \> 300

Definiremos una función para realizar esta clasificación y luego la aplicaremos a la columna `world_rank`.


In [4]:
def create_category(ranking):
    # Como el ranking es un entero, usaremos una serie de sentencias if/elif
    if (ranking >= 1) & (ranking <= 100):
        return "First Tier Top University"
    elif (ranking >= 101) & (ranking <= 200):
        return "Second Tier Top University"
    elif (ranking >= 201) & (ranking <= 300):
        return "Third Tier Top University"
    return "Other Top University"


df['Rank_Level'] = df['world_rank'].apply(lambda x: create_category(x))
df

,world_rank,institution,country,national_rank,quality_of_education,alumni_employment,quality_of_faculty,publications,influence,citations,broad_impact,patents,score,year,Rank_Level
0,1,Harvard University,USA,1,7,9,1,1,1,1,NaN,5,100.00,2012,First Tier Top University
1,2,Massachusetts Institute of Technology,USA,2,9,17,3,12,4,4,NaN,1,91.67,2012,First Tier Top University
2,3,Stanford University,USA,3,17,11,5,4,2,2,NaN,15,89.50,2012,First Tier Top University
3,4,University of Cambridge,United Kingdom,1,10,24,4,16,16,11,NaN,50,86.17,2012,First Tier Top University
4,5,California Institute of Technology,USA,4,2,29,7,37,22,22,NaN,18,85.21,2012,First Tier Top University
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2195,996,University of the Algarve,Portugal,7,367,567,218,926,845,812,969.0,816,44.03,2015,Other Top University
2196,997,Alexandria University,Egypt,4,236,566,218,997,908,645,981.0,871,44.03,2015,Other Top University
2197,998,Federal University of Ceará,Brazil,18,367,549,218,830,823,812,975.0,824,44.03,2015,Other Top University
2198,999,University of A Coruña,Spain,40,367,567,218,886,974,812,975.0,651,44.02,2015,Other Top University


### Creación de una Tabla Pivot Simple

Una tabla pivot nos permite "pivotar" (girar) una columna para que sus valores se conviertan en nuevos encabezados de columna y compararla con otra columna como índice de fila.


Entonces, vamos a crear una tabla que compare el **nivel de ranking** (`Rank_Level`) de las universidades de cada **país** (`country`), utilizando el promedio (`mean`) del **puntaje general** (`score`) por país.

Para esto, indicamos a Pandas:

  * `values`: la columna a agregar (`score`).
  * `index`: la columna que serán las filas (`country`).
  * `columns`: la columna que serán los encabezados (`Rank_Level`).
  * `aggfunc`: la función de agregación (`mean`).


In [5]:
df.pivot_table(values='score',
               index='country',
               columns='Rank_Level',
               aggfunc='mean')

Rank_Level,First Tier Top University,Other Top University,Second Tier Top University,Third Tier Top University
country,,,,
Argentina,NaN,44.672857,NaN,NaN
Australia,47.942500,44.645750,49.242500,47.285000
Austria,NaN,44.864286,NaN,47.066667
Belgium,51.875000,45.081000,49.084000,46.746667
Brazil,NaN,44.499706,49.565000,NaN
Bulgaria,NaN,44.335000,NaN,NaN
Canada,53.633846,44.760541,49.218182,46.826364
Chile,NaN,44.767500,NaN,NaN
China,53.592500,44.564267,47.868000,46.926250


> Observamos un DataFrame donde en el índice (filas) están los paises y las columnas son los niveles en el ranking. Los valores son la **puntuación promedio** para esa combinación de país y nivel de ranking. Los valores **NaN** (Not a Number) indican que no hay observaciones para esa combinación, como en el caso de Argentina, que solo tiene universidades en la categoría "Other Top University".


### Múltiples Funciones de Agregación

Las tablas pivot no se limitan a una única función. Podemos pasar una **lista de funciones** al parámetro `aggfunc`. Pandas proporcionará el resultado utilizando **nombres de columnas jerárquicas** (MultiIndex).

Intentemos la misma consulta, pero agregando también la función `max` (puntuación máxima).

In [6]:
df.pivot_table(values='score',
               index='country',
               columns='Rank_Level',
               aggfunc=['mean', 'max'])

mean                       \
Rank_Level           First Tier Top University Other Top University   
country                                                               
Argentina                                  NaN            44.672857   
Australia                            47.942500            44.645750   
Austria                                    NaN            44.864286   
Belgium                              51.875000            45.081000   
Brazil                                     NaN            44.499706   
Bulgaria                                   NaN            44.335000   
Canada                               53.633846            44.760541   
Chile                                      NaN            44.767500   
China                                53.592500            44.564267   
Colombia                                   NaN            44.432500   
Croatia                                    NaN            44.770000   
Cyprus                                     NaN            44.210000   
Czech Republic                             NaN            44.587778   
Denmark                              49.180000            45.177500   
Egypt                                      NaN            44.230000   
Estonia                                    NaN            44.810000   
Finland                              44.415000            45.062500   
France                               51.914444            44.609028   
Germany                              49.153636            44.978305   
Greece                                     NaN            44.854286   
Hong Kong                                  NaN            45.284286   
Hungary                                    NaN            44.603333   
Iceland                                    NaN            44.980000   
India                                      NaN            44.713226   
Iran                                       NaN            44.270000   
Ireland                                    NaN            44.542500   
Israel                               56.307143            45.013333   
Italy                                48.736667            44.964177   
Japan                                58.812692            44.641583   
Lebanon                                    NaN            44.655000   
Lithuania                                  NaN            44.355000   
Malaysia                                   NaN            45.008333   
Mexico                                     NaN            45.102500   
Netherlands                          48.378333            45.163333   
New Zealand                                NaN            44.832000   
Norway                               47.056667            44.950000   
Poland                                     NaN            44.497222   
Portugal                                   NaN            44.717273   
Puerto Rico                                NaN            44.175000   
Romania                                    NaN            44.133333   
Russia                               51.846667            44.462500   
Saudi Arabia                               NaN            44.511250   
Serbia                                     NaN            44.420000   
Singapore                            50.720000                  NaN   
Slovak Republic                            NaN            44.490000   
Slovenia                                   NaN            44.615000   
South Africa                               NaN            45.246667   
South Korea                          55.990000            44.805714   
Spain                                      NaN            44.724730   
Sweden                               50.672000            45.272500   
Switzerland                          54.005000            44.625000   
Taiwan                               54.210000            44.476667   
Thailand                                   NaN            44.830000   
Turkey                                     NaN            44.48100

> Ahora tenemos tanto la media (`mean`) como el máximo (`max`) en las columnas.

### Valores Marginales (`margins`)

Podemos añadir totales generales para resumir los valores de las filas y columnas mediante el parámetro **margins** ajustado a `True`.


Para las columnas de nivel superior (`mean`, `max`), el valor marginal se calculará utilizando la misma función de agregación (es decir, el promedio general para la columna `mean` y el máximo de los máximos para la columna `max`).


In [7]:
df.pivot_table(values='score',
               index='country',
               columns='Rank_Level',
               aggfunc=['mean', 'max'],
               margins=True)

mean                       \
Rank_Level           First Tier Top University Other Top University   
country                                                               
Argentina                                  NaN            44.672857   
Australia                            47.942500            44.645750   
Austria                                    NaN            44.864286   
Belgium                              51.875000            45.081000   
Brazil                                     NaN            44.499706   
Bulgaria                                   NaN            44.335000   
Canada                               53.633846            44.760541   
Chile                                      NaN            44.767500   
China                                53.592500            44.564267   
Colombia                                   NaN            44.432500   
Croatia                                    NaN            44.770000   
Cyprus                                     NaN            44.210000   
Czech Republic                             NaN            44.587778   
Denmark                              49.180000            45.177500   
Egypt                                      NaN            44.230000   
Estonia                                    NaN            44.810000   
Finland                              44.415000            45.062500   
France                               51.914444            44.609028   
Germany                              49.153636            44.978305   
Greece                                     NaN            44.854286   
Hong Kong                                  NaN            45.284286   
Hungary                                    NaN            44.603333   
Iceland                                    NaN            44.980000   
India                                      NaN            44.713226   
Iran                                       NaN            44.270000   
Ireland                                    NaN            44.542500   
Israel                               56.307143            45.013333   
Italy                                48.736667            44.964177   
Japan                                58.812692            44.641583   
Lebanon                                    NaN            44.655000   
Lithuania                                  NaN            44.355000   
Malaysia                                   NaN            45.008333   
Mexico                                     NaN            45.102500   
Netherlands                          48.378333            45.163333   
New Zealand                                NaN            44.832000   
Norway                               47.056667            44.950000   
Poland                                     NaN            44.497222   
Portugal                                   NaN            44.717273   
Puerto Rico                                NaN            44.175000   
Romania                                    NaN            44.133333   
Russia                               51.846667            44.462500   
Saudi Arabia                               NaN            44.511250   
Serbia                                     NaN            44.420000   
Singapore                            50.720000                  NaN   
Slovak Republic                            NaN            44.490000   
Slovenia                                   NaN            44.615000   
South Africa                               NaN            45.246667   
South Korea                          55.990000            44.805714   
Spain                                      NaN            44.724730   
Sweden                               50.672000            45.272500   
Switzerland                          54.005000            44.625000   
Taiwan                               54.210000            44.476667   
Thailand                                   NaN            44.830000   
Turkey                                     NaN            44.48100

> En la columna **'All'** de cada fila ahora se proporcionan los totales generales (agregaciones).

### Consulta de Datos en Tablas Pivot Jerárquicas

Una tabla pivot es solo un **DataFrame de múltiples niveles** (MultiIndex). Por lo tanto, podemos acceder a sus datos de forma similar a un DataFrame regular.

In [8]:
new_df = df.pivot_table(values='score',
                        index='country',
                        columns='Rank_Level',
                        aggfunc=['mean', 'max'],
                        margins=True)

# Veamos el índice
new_df.index

Index(['Argentina', 'Australia', 'Austria', 'Belgium', 'Brazil', 'Bulgaria',
       'Canada', 'Chile', 'China', 'Colombia', 'Croatia', 'Cyprus',
       'Czech Republic', 'Denmark', 'Egypt', 'Estonia', 'Finland', 'France',
       'Germany', 'Greece', 'Hong Kong', 'Hungary', 'Iceland', 'India', 'Iran',
       'Ireland', 'Israel', 'Italy', 'Japan', 'Lebanon', 'Lithuania',
       'Malaysia', 'Mexico', 'Netherlands', 'New Zealand', 'Norway', 'Poland',
       'Portugal', 'Puerto Rico', 'Romania', 'Russia', 'Saudi Arabia',
       'Serbia', 'Singapore', 'Slovak Republic', 'Slovenia', 'South Africa',
       'South Korea', 'Spain', 'Sweden', 'Switzerland', 'Taiwan', 'Thailand',
       'Turkey', 'USA', 'Uganda', 'United Arab Emirates', 'United Kingdom',
       'Uruguay', 'All'],
      dtype='object', name='country')

In [9]:
# Columnas
new_df.columns

MultiIndex([('mean',  'First Tier Top University'),
            ('mean',       'Other Top University'),
            ('mean', 'Second Tier Top University'),
            ('mean',  'Third Tier Top University'),
            ('mean',                        'All'),
            ( 'max',  'First Tier Top University'),
            ( 'max',       'Other Top University'),
            ( 'max', 'Second Tier Top University'),
            ( 'max',  'Third Tier Top University'),
            ( 'max',                        'All')],
           names=[None, 'Rank_Level'])

Para obtener los valores **promedio** (`mean`) de las universidades de **Primer Nivel** (`First Tier Top University`) en cada país, realizamos una doble proyección:

In [10]:
new_df['mean']['First Tier Top University']

country
Argentina                     NaN
Australia               47.942500
Austria                       NaN
Belgium                 51.875000
Brazil                        NaN
Bulgaria                      NaN
Canada                  53.633846
Chile                         NaN
China                   53.592500
Colombia                      NaN
Croatia                       NaN
Cyprus                        NaN
Czech Republic                NaN
Denmark                 49.180000
Egypt                         NaN
Estonia                       NaN
Finland                 44.415000
France                  51.914444
Germany                 49.153636
Greece                        NaN
Hong Kong                     NaN
Hungary                       NaN
Iceland                       NaN
India                         NaN
Iran                          NaN
Ireland                       NaN
Israel                  56.307143
Italy                   48.736667
Japan                   58.812692
Lebano

El resultado es un objeto **Series**, lo que podemos confirmar con la función `type()`

In [11]:
type(new_df['mean']['First Tier Top University'])

pandas.core.series.Series

#### Encontrar el País con el Máximo Promedio

Podemos usar la función **`idxmax()`** para encontrar el índice (en este caso, el país) que tiene el máximo valor en la Serie de promedios de "First Tier Top University".

In [12]:
new_df['mean']['First Tier Top University'].idxmax()

'United Kingdom'

### Reestructuración con `stack()` y `unstack()`

Las funciones `stack()` y `unstack()` nos permiten cambiar la forma de la tabla pivote.

  * **`stack()`**: Pivota (intercambia) el índice de columna de nivel más bajo para que se convierta en el índice de fila más interno.
  * **`unstack()`**: Es la operación inversa, pívota el índice de fila más interno para que se convierta en el índice de columna de nivel más bajo.

#### Usando `stack()`

Veamos la forma original de nuestro DataFrame pivot antes de aplicar `stack()`:

In [13]:
new_df.head()

mean                       \
Rank_Level First Tier Top University Other Top University   
country                                                     
Argentina                        NaN            44.672857   
Australia                    47.9425            44.645750   
Austria                          NaN            44.864286   
Belgium                      51.8750            45.081000   
Brazil                           NaN            44.499706   

                                                                            \
Rank_Level Second Tier Top University Third Tier Top University        All   
country                                                                      
Argentina                         NaN                       NaN  44.672857   
Australia                     49.2425                 47.285000  45.825517   
Austria                           NaN                 47.066667  45.139583   
Belgium                       49.0840                 46.746667  47.011000   
Brazil                        49.5650                       NaN  44.781111   

                                 max                       \
Rank_Level First Tier Top University Other Top University   
country                                                     
Argentina                        NaN                45.66   
Australia                      51.61                45.97   
Austria                          NaN                46.29   
Belgium                        52.03                46.21   
Brazil                           NaN                46.08   

                                                                        
Rank_Level Second Tier Top University Third Tier Top University    All  
country                                                                 
Argentina                         NaN                       NaN  45.66  
Australia                       50.40                     47.47  51.61  
Austria                           NaN                     47.78  47.78  
Belgium                         49.73                     47.14  52.03  
Brazil                          49.82                       NaN  49.82

Al aplicar `stack()`, el nivel de columna más bajo (`Rank_Level`) se convierte en el índice de fila más interno, a la derecha de `country`.

In [14]:
new_df = new_df.stack(future_stack=True)

new_df.head()

mean    max
country   Rank_Level                                  
Argentina First Tier Top University         NaN    NaN
          Other Top University        44.672857  45.66
          Second Tier Top University        NaN    NaN
          Third Tier Top University         NaN    NaN
          All                         44.672857  45.66

#### Usando `unstack()`

Al aplicar `unstack()` en el DataFrame apilado, restauramos su forma original, moviendo el índice de fila más interno (`Rank_Level`) de vuelta a las columnas.


In [15]:
new_df.unstack().head()

mean                       \
Rank_Level First Tier Top University Other Top University   
country                                                     
Argentina                        NaN            44.672857   
Australia                    47.9425            44.645750   
Austria                          NaN            44.864286   
Belgium                      51.8750            45.081000   
Brazil                           NaN            44.499706   

                                                                            \
Rank_Level Second Tier Top University Third Tier Top University        All   
country                                                                      
Argentina                         NaN                       NaN  44.672857   
Australia                     49.2425                 47.285000  45.825517   
Austria                           NaN                 47.066667  45.139583   
Belgium                       49.0840                 46.746667  47.011000   
Brazil                        49.5650                       NaN  44.781111   

                                 max                       \
Rank_Level First Tier Top University Other Top University   
country                                                     
Argentina                        NaN                45.66   
Australia                      51.61                45.97   
Austria                          NaN                46.29   
Belgium                        52.03                46.21   
Brazil                           NaN                46.08   

                                                                        
Rank_Level Second Tier Top University Third Tier Top University    All  
country                                                                 
Argentina                         NaN                       NaN  45.66  
Australia                       50.40                     47.47  51.61  
Austria                           NaN                     47.78  47.78  
Belgium                         49.73                     47.14  52.03  
Brazil                          49.82                       NaN  49.82

Al realizar **`unstack()`** dos veces seguidas, se despliegan todos los niveles jerárquicos de las filas en las columnas, dando como resultado un objeto **Series** con un MultiIndex complejo, pero con una sola columna de valores:

In [16]:
new_df.unstack().unstack().head()

      Rank_Level                 country  
mean  First Tier Top University  Argentina        NaN
                                 Australia    47.9425
                                 Austria          NaN
                                 Belgium      51.8750
                                 Brazil           NaN
dtype: float64

## Conclusión

Las tablas pivot son increíblemente útiles para trabajar con datos numéricos, especialmente al intentar **resumir los datos** de alguna forma.

Normalmente, crearás nuevas tablas pivot en secciones de datos, ya sea que estés explorando los datos o preparando datos para reportes.

Recuerda que puedes pasar **cualquier función** al parámetro `aggfunc`, incluidas las que definas tú mismo.